# Exploratory Data Analysis — GTZAN Dataset

This notebook explores the GTZAN dataset used for our Music Genre Classification project.
We examine the distribution of audio features, visualize mel-spectrograms, and check data quality.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

import config

sns.set_theme(style='darkgrid')
%matplotlib inline

## 1. Load Feature CSVs

In [ ]:
df_30s = pd.read_csv(config.CSV_30S)
df_3s = pd.read_csv(config.CSV_3S)

print(f"30-sec features: {df_30s.shape}")
print(f"3-sec features:  {df_3s.shape}")
df_30s.head()

## 2. Genre Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_30s['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of Tracks per Genre (30-sec)')
ax.set_xlabel('Genre')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Feature Distributions — Key Audio Descriptors

In [ ]:
key_features = ['spectral_centroid_mean', 'zero_crossing_rate_mean',
                'mfcc1_mean', 'mfcc2_mean', 'tempo', 'rms_mean']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, feat in enumerate(key_features):
    ax = axes[i // 3, i % 3]
    for genre in config.GENRES:
        subset = df_30s[df_30s['label'] == genre]
        ax.hist(subset[feat], bins=20, alpha=0.5, label=genre)
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

axes[0, 0].legend(fontsize=6)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix of Key Features

In [ ]:
numeric_cols = df_30s.select_dtypes(include=[np.number]).columns.tolist()
if 'length' in numeric_cols:
    numeric_cols.remove('length')

corr = df_30s[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
            xticklabels=True, yticklabels=True, fmt='.1f')
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Visualize Audio Waveforms and Mel-Spectrograms

In [ ]:
# Pick one sample from each genre
fig, axes = plt.subplots(5, 4, figsize=(20, 20))

for i, genre in enumerate(config.GENRES):
    wav_path = os.path.join(config.AUDIO_DIR, genre, f"{genre}.00000.wav")
    y, sr = librosa.load(wav_path, sr=config.SAMPLE_RATE)

    # Waveform
    row, col = (i // 2) , (i % 2) * 2
    ax_wave = axes[row, col]
    librosa.display.waveshow(y, sr=sr, ax=ax_wave, color='steelblue')
    ax_wave.set_title(f"{genre} — Waveform")

    # Mel-spectrogram
    ax_mel = axes[row, col + 1]
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, hop_length=512,
                             x_axis='time', y_axis='mel', ax=ax_mel)
    ax_mel.set_title(f"{genre} — Mel-Spectrogram")

plt.tight_layout()
plt.show()

## 6. Feature Statistics Summary

In [ ]:
# Summary statistics grouped by genre
summary = df_30s.groupby('label')[key_features].agg(['mean', 'std'])
summary.columns = ['_'.join(col) for col in summary.columns]
summary.round(3)

## 7. Check for Missing Values and Data Quality

In [ ]:
print("Missing values in 30-sec CSV:")
print(df_30s.isnull().sum().sum())

print("\nMissing values in 3-sec CSV:")
print(df_3s.isnull().sum().sum())

print(f"\n3-sec rows per genre:")
print(df_3s['label'].value_counts().sort_index())